# Calibrated Prediction Intervals for Nuclear Masses under Extrapolation — v2

**What changed from v1**

1. **More data for evaluation.** AME2003 is added, giving several *rolling temporal folds* (train on one edition, test on nuclei first measured in a later one). Results are pooled across folds.
2. **Peel-CQR (new method).** Calibration scores come from *frontier peeling*: the outer layers of the training chart are held out, the quantile model is fitted on the remaining core, and the held-out nuclei — hundreds at each distance — serve as calibration points at known extrapolation distances. No later AME edition is needed for calibration.
3. **Statistical inference on coverage.** Clopper–Pearson 95% intervals for every coverage, exact binomial tests against the nominal 90%, and exact McNemar tests for paired method comparisons.
4. **Point prediction = median quantile model**, so the point always lies inside its interval.
5. **Robustness by subsampling** (80% without replacement) instead of the bootstrap.
6. **Pluggable physics base model** (WS4, FRDM2012, HFB-24…) through a simple loader.

**Methods compared**

| Method | Calibration data | Distance-aware |
|---|---|---|
| GP | Bayesian posterior | implicitly |
| CQR-random | random 20% of training nuclei | no |
| CQR-temporal | nuclei first measured in the next AME edition | no |
| CQR-temporal-scaled | next edition, scores scaled by g(d) learned from peeling | yes |
| **Peel-CQR** | peeled frontier of the training set, per distance | yes |

Runs on CPU (~5–10 min). On Kaggle turn **Internet ON**, or upload the AME files as a dataset.

In [ ]:
import os, glob, shutil, time, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
ALPHA = 0.10            # 90% intervals
K_PEEL = 3              # number of frontier layers peeled off
MIN_Z, MIN_N = 8, 8
RUN_GP = True           # set False to save ~1 min per fold
N_SUBSAMPLE = 10        # robustness repetitions
DATA_DIR = "ame_data"
os.makedirs(DATA_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Get the AME tables (2003, 2012, 2016, 2020)

Order tried: a copy in `ame_data/` → any matching file under `/kaggle/input` (uploaded dataset) → several mirrors. AME2003 is optional: without it the notebook still runs with the single 2012→2016→2020 fold.

Manual download page: https://www-nds.iaea.org/amdc/ — files `mass.mas03`, `mass.mas12`, `mass16.txt`, `mass_1.mas20.txt`.

In [ ]:
AME_URLS = {
    2003: ["https://www-nds.iaea.org/amdc/ame2003/mass.mas03",
           "http://amdc.in2p3.fr/masstables/Ame2003/mass.mas03",
           "https://amdc.impcas.ac.cn/masstables/Ame2003/mass.mas03"],
    2012: ["https://raw.githubusercontent.com/yamenetoo/nuclear-mass-peel-cqr/main/data/ame/mass.mas12",
           "https://www-nds.iaea.org/amdc/ame2012/mass.mas12",
           "http://amdc.in2p3.fr/masstables/Ame2012/mass.mas12",
           "https://amdc.impcas.ac.cn/masstables/Ame2012/mass.mas12"],
    2016: ["https://www-nds.iaea.org/amdc/ame2016/mass16.txt",
           "http://amdc.in2p3.fr/masstables/Ame2016/mass16.txt",
           "https://amdc.impcas.ac.cn/masstables/Ame2016/mass16.txt"],
    2020: ["https://raw.githubusercontent.com/yamenetoo/nuclear-mass-peel-cqr/main/data/ame/mass_1.mas20.txt",
           "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20.txt",
           "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20",
           "https://amdc.impcas.ac.cn/masstables/Ame2020/mass_1.mas20.txt"],
}
AME_FILES = {2003: "mass.mas03", 2012: "mass.mas12", 2016: "mass16.txt", 2020: "mass_1.mas20.txt"}
ALT_NAMES = {2003: ["mass.mas03", "mass.mas03.txt", "mass03.txt"],
             2012: ["mass.mas12", "mass.mas12.txt", "mass12.txt"],
             2016: ["mass16.txt", "mass.mas16", "mass16"],
             2020: ["mass_1.mas20.txt", "mass_1.mas20", "mass.mas20"]}
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
           "Accept": "text/plain,text/html;q=0.9,*/*;q=0.8",
           "Referer": "https://www-nds.iaea.org/amdc/"}


def fetch(url):
    try:
        import requests
        r = requests.get(url, headers=HEADERS, timeout=60); r.raise_for_status(); return r.content
    except ImportError:
        with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=60) as r:
            return r.read()


def get_ame(year):
    path = os.path.join(DATA_DIR, AME_FILES[year])
    if os.path.exists(path) and os.path.getsize(path) > 100_000:
        print(f"AME{year}: found {path}"); return path
    for root in ["/kaggle/input", "data/ame", "../data/ame", ".", os.path.expanduser("~")]:
        for name in ALT_NAMES[year]:
            hits = [h for h in glob.glob(os.path.join(root, "**", name), recursive=True)
                    if os.path.getsize(h) > 100_000 and os.path.abspath(h) != os.path.abspath(path)]
            if hits:
                shutil.copy(hits[0], path); print(f"AME{year}: copied from {hits[0]}"); return path
    for url in AME_URLS[year]:
        try:
            data = fetch(url)
            if len(data) < 100_000:
                raise ValueError(f"only {len(data)} bytes")
            open(path, "wb").write(data); print(f"AME{year}: downloaded from {url}"); return path
        except Exception as e:
            print(f"AME{year}: {url} -> {e}")
    return None


paths = {y: get_ame(y) for y in (2003, 2012, 2016, 2020)}
missing = [y for y in (2012, 2016, 2020) if paths[y] is None]
if missing:
    raise FileNotFoundError(f"Required AME editions missing: {missing}. See the links above.")
EDITIONS = [y for y in (2003, 2012, 2016, 2020) if paths[y] is not None]
print("editions available:", EDITIONS)

## 2. Parse

Leading columns are identical in all editions: `N`, `Z`, `A` in characters 5–19 and the mass excess Δ (keV) starting at character 29. A `#` marks a value estimated from systematics.

$$B = Z\,\Delta_{^1\mathrm{H}} + N\,\Delta_n - \Delta,\qquad \Delta_{^1\mathrm{H}} = 7288.971064,\ \Delta_n = 8071.318062\ \mathrm{keV}.$$

In [ ]:
DELTA_H, DELTA_N = 7288.971064, 8071.318062


def parse_ame(path):
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            if len(line) < 40:
                continue
            try:
                N = int(line[4:9]); Z = int(line[9:14]); A = int(line[14:19])
            except ValueError:
                continue
            if N + Z != A or A < 1:
                continue
            toks = line[28:].split()
            if not toks:
                continue
            try:
                me = float(toks[0].replace("#", ""))
            except ValueError:
                continue
            rows.append((Z, N, A, me, "#" in toks[0]))
    df = pd.DataFrame(rows, columns=["Z", "N", "A", "mass_excess", "estimated"]).drop_duplicates(["Z", "N"])
    df["B"] = df.Z * DELTA_H + df.N * DELTA_N - df.mass_excess
    return df.reset_index(drop=True)


ame = {y: parse_ame(paths[y]) for y in EDITIONS}
for y, d in ame.items():
    print(f"AME{y}: {len(d):5d} rows | measured {(~d.estimated).sum():5d} | estimated {d.estimated.sum():5d}")

d20 = ame[2020]
for z_, a_ in [(26, 56), (82, 208)]:
    r = d20[(d20.Z == z_) & (d20.A == a_)]
    print(f"check Z={z_} A={a_}: B/A = {r.B.iloc[0] / a_:.3f} keV (expected ~8790 / ~7867)" if len(r)
          else f"check Z={z_} A={a_}: missing -> inspect parser")

In [ ]:
def measured(y):
    d = ame[y]; m = d[(~d.estimated) & (d.Z >= MIN_Z) & (d.N >= MIN_N)]
    return set(zip(m.Z, m.N))


MEAS = {y: measured(y) for y in EDITIONS}
data = d20[(d20.Z >= MIN_Z) & (d20.N >= MIN_N)].copy().reset_index(drop=True)
data["key"] = list(zip(data.Z, data.N))
data["measured2020"] = data.key.map(lambda k: k in MEAS[2020])
data["first_measured"] = data.key.map(lambda k: next((y for y in EDITIONS if k in MEAS[y]), np.nan))
print(data.groupby("first_measured", dropna=False).size().rename("nuclei"))

## 3. Physics base model

Default: a liquid-drop formula **refitted on each fold's training set** (no leakage).

$$B_{\mathrm{LDM}} = a_v A - a_s A^{2/3} - a_c \tfrac{Z(Z-1)}{A^{1/3}} - a_a \tfrac{(N-Z)^2}{A} + a_{ss}\tfrac{(N-Z)^2}{A^{4/3}} + a_p\,\delta\,A^{-1/2}$$

**Using WS4 / FRDM2012 / HFB-24.** Download the model's table, then call `load_base_table(...)` with the column positions of Z, N and the predicted mass excess or binding energy. Nuclei missing from that table are dropped. A strong base model is the single biggest way to shrink the intervals.

In [ ]:
def load_base_table(path, z_col, n_col, value_col, kind="mass_excess", unit="MeV", skiprows=0, sep=r"\s+"):
    # kind: 'mass_excess' or 'binding' (total binding energy); unit: 'MeV' or 'keV'
    t = pd.read_csv(path, sep=sep, header=None, skiprows=skiprows, comment="#", engine="python")
    out = pd.DataFrame({"Z": t.iloc[:, z_col].astype(int), "N": t.iloc[:, n_col].astype(int),
                        "v": t.iloc[:, value_col].astype(float) * (1000.0 if unit == "MeV" else 1.0)})
    if kind == "mass_excess":
        out["B_theory"] = out.Z * DELTA_H + out.N * DELTA_N - out.v
    else:
        out["B_theory"] = out.v
    return out[["Z", "N", "B_theory"]]


BASE_TABLE = None
# Example (adjust columns to your file):
# BASE_TABLE = load_base_table("ws4.txt", z_col=0, n_col=1, value_col=3, kind="mass_excess", unit="MeV", skiprows=1)
if BASE_TABLE is not None:
    data = data.merge(BASE_TABLE, on=["Z", "N"], how="inner").reset_index(drop=True)
    print("nuclei covered by base table:", len(data))


def ldm_design(Z, N):
    Z = np.asarray(Z, float); N = np.asarray(N, float); A = Z + N
    dlt = np.where((Z % 2 == 0) & (N % 2 == 0), 1.0, np.where((Z % 2 == 1) & (N % 2 == 1), -1.0, 0.0))
    return np.column_stack([A, -A ** (2 / 3), -Z * (Z - 1) / A ** (1 / 3), -(N - Z) ** 2 / A,
                            (N - Z) ** 2 / A ** (4 / 3), dlt * A ** -0.5])


def base_prediction(train_mask):
    # returns B_theory (keV) for every row of `data`
    if BASE_TABLE is not None:
        return data.B_theory.values
    coef, *_ = np.linalg.lstsq(ldm_design(data.Z[train_mask], data.N[train_mask]), data.B[train_mask], rcond=None)
    return ldm_design(data.Z, data.N) @ coef

## 4. Features, distances and frontier peeling

* Features: Z, N, A, isospin asymmetry, parities, valence numbers, shell-filling fractions, Casten factor $P$.
* $d$ = Manhattan distance $|\Delta Z|+|\Delta N|$ to the nearest training nucleus.
* **Depth** of a training nucleus = distance to the nearest *unmeasured* position. Peeling layer $k$ = training nuclei with depth $\le K$; the **core** has depth $> K$. The distance of a peeled nucleus to the core plays the role of $d$.

In [ ]:
MAGIC = np.array([2, 8, 20, 28, 50, 82, 126, 184])


def valence(x):
    x = np.asarray(x)[:, None]
    lower = np.where(MAGIC[None, :] <= x, MAGIC[None, :], -1).max(1)
    upper = np.where(MAGIC[None, :] > x, MAGIC[None, :], 10 ** 6).min(1)
    xv = x[:, 0]
    return np.minimum(xv - lower, upper - xv), (xv - lower) / (upper - lower)


def features(df):
    Z, N = df.Z.values.astype(float), df.N.values.astype(float); A = Z + N
    vz, fz = valence(df.Z.values); vn, fn = valence(df.N.values)
    P = np.where(vz + vn > 0, vz * vn / np.maximum(vz + vn, 1), 0.0)
    return np.column_stack([Z, N, A, (N - Z) / A, Z % 2, N % 2, vz, vn, fz, fn, P])


X_ALL = features(data)
ZN = data[["Z", "N"]].values


def manhattan_to(ref, pts):
    ref = np.asarray(ref); pts = np.asarray(pts)
    out = np.empty(len(pts), int)
    for i in range(0, len(pts), 512):
        p = pts[i:i + 512]
        out[i:i + 512] = (np.abs(p[:, None, 0] - ref[None, :, 0]) + np.abs(p[:, None, 1] - ref[None, :, 1])).min(1)
    return out


def depth_in(set_zn):
    # Manhattan distance of each member to the nearest grid position (Z,N >= MIN-1) not in the set
    zn = np.asarray(set_zn)
    zmax, nmax = zn[:, 0].max() + K_PEEL + 2, zn[:, 1].max() + K_PEEL + 2
    grid = np.zeros((zmax + 1, nmax + 1), bool); grid[zn[:, 0], zn[:, 1]] = True
    holes = np.argwhere(~grid)
    holes = holes[(holes[:, 0] >= MIN_Z - 1) & (holes[:, 1] >= MIN_N - 1)]
    return manhattan_to(holes, zn)

## 5. Learners and conformal tools

* `QuantileGB` fits three gradient-boosted models (α/2, 0.5, 1−α/2) and sorts them, so lo ≤ median ≤ hi. **Any model with `fit(X, y)` and `predict3(X) -> (lo, med, hi)` can replace it — e.g. your NDT quantile network.**
* Conformal quantile: $\hat c = $ the $\lceil (n+1)(1-\alpha)\rceil$-th smallest score.
* Coverage uncertainty: Clopper–Pearson; tests: exact binomial and exact McNemar.

In [ ]:
GB_KW = dict(max_iter=600, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=8,
             l2_regularization=1.0, random_state=SEED)


class QuantileGB:
    def __init__(self, alpha=ALPHA, **kw):
        p = dict(GB_KW, **kw)
        self.ms = [HistGradientBoostingRegressor(loss="quantile", quantile=q, **p)
                   for q in (alpha / 2, 0.5, 1 - alpha / 2)]

    def fit(self, X, y):
        for m in self.ms:
            m.fit(X, y)
        return self

    def predict3(self, X):
        P = np.sort(np.column_stack([m.predict(X) for m in self.ms]), axis=1)
        return P[:, 0], P[:, 1], P[:, 2]


def cqr_scores(model, X, y):
    lo, _, hi = model.predict3(X)
    return np.maximum(lo - y, y - hi)


def conformal_q(s, alpha=ALPHA):
    s = np.sort(np.asarray(s)); n = len(s)
    if n == 0:
        return np.nan
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return s[min(k, n) - 1] if k <= n else s[-1]   # k > n: infinite in theory; use max (flagged by small n)


def clopper_pearson(k, n, level=0.95):
    if n == 0:
        return np.nan, np.nan
    a = 1 - level
    lo = 0.0 if k == 0 else stats.beta.ppf(a / 2, k, n - k + 1)
    hi = 1.0 if k == n else stats.beta.ppf(1 - a / 2, k + 1, n - k)
    return lo, hi


def binom_p(k, n, p0=1 - ALPHA):
    return stats.binomtest(k, n, p0).pvalue if n else np.nan


def mcnemar_p(cov_a, cov_b):
    b = int(np.sum(cov_a & ~cov_b)); c = int(np.sum(~cov_a & cov_b))
    return (stats.binomtest(b, b + c, 0.5).pvalue if b + c else 1.0), b, c


Z90 = stats.norm.ppf(1 - ALPHA / 2)

## 6. One fold = (train edition, calibration edition, test edition)

For a fold `(e_tr, e_cal, e_te)`:

* **train** = measured in `e_tr`
* **calib** = first measured in `e_cal` (not in `e_tr`) — only used by the two temporal methods
* **test** = measured in `e_te` but not in `e_cal` (or not in `e_tr` if there is no calibration edition)

All targets come from AME2020. Peel-CQR and γ are learned from the training set alone.

In [ ]:
def learn_peel(Xtr, ytr, zn_tr, alpha=ALPHA, K=None, learner=QuantileGB, domain_zn=None):
    # domain_zn: the measured region whose frontier is peeled (defaults to the training nuclei themselves;
    # pass the full edition when the training set is a random subsample, so random holes are not "frontier")
    K = K_PEEL if K is None else K
    dom = zn_tr if domain_zn is None else np.asarray(domain_zn)
    dep_dom = dict(zip(map(tuple, dom), depth_in(dom)))
    dep = np.array([dep_dom.get(tuple(p), 1) for p in zn_tr])
    core = dep > K
    if core.sum() < 50:
        raise ValueError(f"core too small ({core.sum()}) for K={K}")
    qm = learner(alpha).fit(Xtr[core], ytr[core])
    d_core = manhattan_to(zn_tr[core], zn_tr[~core])
    s = cqr_scores(qm, Xtr[~core], ytr[~core])
    ds = np.arange(1, K + 1)
    n_d = np.array([np.sum(d_core == d) for d in ds])
    c_d = np.array([conformal_q(s[d_core == d], alpha) for d in ds])
    ok = n_d >= 20
    # linear growth for the scaled method and for distances beyond K
    if ok.sum() >= 2:
        slope, icpt = np.polyfit(ds[ok], c_d[ok], 1)
        slope = max(slope, 0.0)
        r = c_d[ok] / c_d[ok][0]
        gamma = max(0.0, float(np.sum((ds[ok] - 1) * (r - 1)) / np.sum((ds[ok] - 1) ** 2)))
    else:
        slope, gamma = 0.0, 0.0
    c_d = pd.Series(np.where(ok, c_d, np.nan)).ffill().bfill().cummax().values
    return dict(core=core, n_core=int(core.sum()), n_d=n_d, c_d=c_d, slope=slope, gamma=gamma,
                scores=s, d_core=d_core)


def peel_c(peel, d):
    d = np.maximum(np.asarray(d), 1); K = len(peel["c_d"])
    inside = peel["c_d"][np.minimum(d, K) - 1]
    return np.where(d <= K, inside, peel["c_d"][-1] + peel["slope"] * (d - K))


def run_fold(e_tr, e_cal, e_te, learner=QuantileGB, gb_kw=None, run_gp=RUN_GP, sub_idx=None, verbose=True):
    keys = data.key.values
    tr = np.array([k in MEAS[e_tr] for k in keys]) & data.measured2020.values
    domain = ZN[tr]                              # full measured region of the training edition
    if sub_idx is not None:                      # subsampling for robustness
        tr_idx = np.flatnonzero(tr); keep = np.zeros_like(tr); keep[tr_idx[sub_idx]] = True; tr = keep
    prev = MEAS[e_cal] if e_cal else MEAS[e_tr]
    cal = (np.array([(k in MEAS[e_cal]) and (k not in MEAS[e_tr]) for k in keys]) & data.measured2020.values
           if e_cal else np.zeros(len(data), bool))
    te = np.array([(k in MEAS[e_te]) and (k not in prev) and (k not in MEAS[e_tr]) for k in keys]) & data.measured2020.values

    yv = (data.B.values - base_prediction(tr)) / 1000.0          # residual, MeV
    Xtr, ytr, zn_tr = X_ALL[tr], yv[tr], ZN[tr]
    d_te = manhattan_to(zn_tr, ZN[te]); d_cal = manhattan_to(zn_tr, ZN[cal]) if cal.any() else np.array([], int)
    mk = (lambda: learner(ALPHA, **gb_kw)) if gb_kw else (lambda: learner(ALPHA))

    qm = mk().fit(Xtr, ytr)
    lo, med, hi = qm.predict3(X_ALL[te])
    out = {}

    # CQR-random
    i_fit, i_cal = train_test_split(np.arange(len(ytr)), test_size=0.2, random_state=SEED)
    qr = mk().fit(Xtr[i_fit], ytr[i_fit])
    c_r = conformal_q(cqr_scores(qr, Xtr[i_cal], ytr[i_cal]))
    lr, _, hr = qr.predict3(X_ALL[te]); out["CQR-random"] = (lr - c_r, hr + c_r)

    # peeling
    peel = learn_peel(Xtr, ytr, zn_tr, learner=(lambda a: mk()), domain_zn=domain)
    g = lambda d: 1.0 + peel["gamma"] * (np.maximum(d, 1) - 1)
    cp = peel_c(peel, d_te); out["Peel-CQR"] = (lo - cp, hi + cp)

    # temporal methods
    c_t = c_s = np.nan
    if cal.any():
        s_cal = cqr_scores(qm, X_ALL[cal], yv[cal])
        c_t = conformal_q(s_cal); c_s = conformal_q(s_cal / g(d_cal))
        out["CQR-temporal"] = (lo - c_t, hi + c_t)
        out["CQR-temporal-scaled"] = (lo - c_s * g(d_te), hi + c_s * g(d_te))

    # GP
    if run_gp:
        cols = [0, 1, 4, 5]; mu, sd = Xtr[:, cols].mean(0), Xtr[:, cols].std(0)
        gp = GaussianProcessRegressor(C(1.0) * RBF([0.2, 0.2, 1.0, 1.0]) + WhiteKernel(0.1),
                                      normalize_y=True, random_state=SEED).fit((Xtr[:, cols] - mu) / sd, ytr)
        m_, s_ = gp.predict((X_ALL[te][:, cols] - mu) / sd, return_std=True)
        out["GP"] = (m_ - Z90 * s_, m_ + Z90 * s_)

    fold = f"{e_tr}->{e_cal or '-'}->{e_te}"
    if verbose:
        print(f"fold {fold}: train {tr.sum()}, calib {cal.sum()}, test {te.sum()} | "
              f"peel core {peel['n_core']}, n_d {peel['n_d'].tolist()}, gamma {peel['gamma']:.3f}")
    ytest = yv[te]
    pred = pd.DataFrame({"fold": fold, "Z": data.Z.values[te], "N": data.N.values[te], "d": d_te,
                         "y_MeV": ytest, "median_MeV": med})
    for name, (a, b) in out.items():
        pred[f"{name}_lo"] = a; pred[f"{name}_hi"] = b
        pred[f"{name}_cov"] = (ytest >= a) & (ytest <= b)
    info = dict(fold=fold, n_train=int(tr.sum()), n_calib=int(cal.sum()), n_test=int(te.sum()),
                c_random=c_r, c_temporal=c_t, c_scaled=c_s, gamma=peel["gamma"], peel_slope=peel["slope"],
                **{f"peel_c_d{d+1}": v for d, v in enumerate(peel["c_d"])},
                **{f"peel_n_d{d+1}": v for d, v in enumerate(peel["n_d"])})
    return pred, info, list(out.keys())

## 7. Run all folds

In [ ]:
FOLDS = []
if 2003 in EDITIONS:
    FOLDS += [(2003, 2012, 2016)]
FOLDS += [(2012, 2016, 2020)]
if 2003 in EDITIONS:
    FOLDS += [(2003, None, 2012)]      # extra fold for methods that need no later calibration edition

t0 = time.time()
preds, infos = [], []
for f in FOLDS:
    p, i, _ = run_fold(*f); preds.append(p); infos.append(i)
pred_all = pd.concat(preds, ignore_index=True)
fold_info = pd.DataFrame(infos)
print(f"done in {time.time() - t0:.0f}s; pooled test nuclei: {len(pred_all)}")
METHODS = ["GP", "CQR-random", "CQR-temporal", "CQR-temporal-scaled", "Peel-CQR"]
METHODS = [m for m in METHODS if f"{m}_cov" in pred_all]
fold_info.round(3)

In [ ]:
def summarize(df, by=None):
    rows = []
    groups = [("all", df)] if by is None else list(df.groupby(by))
    for gname, g in groups:
        for m in METHODS:
            col = f"{m}_cov"
            if col not in g:
                continue
            v = g[col].dropna().astype(bool)
            n, k = len(v), int(v.sum())
            if n == 0:
                continue
            w = (g[f"{m}_hi"] - g[f"{m}_lo"]).dropna()
            lo, hi = clopper_pearson(k, n)
            rows.append({(by or "group"): gname, "method": m, "n": n, "coverage": k / n,
                         "CP95_lo": lo, "CP95_hi": hi, "p_vs_0.90": binom_p(k, n),
                         "mean_width_MeV": w.mean(), "median_width_MeV": w.median()})
    return pd.DataFrame(rows)


per_fold = summarize(pred_all, by="fold")
pooled = summarize(pred_all)
print(f"target coverage {1 - ALPHA:.2f}\n")
print("Pooled over all folds (each method on the folds where it is defined):")
pooled.round(3)

In [ ]:
per_fold.round(3)

## 8. Coverage and width versus extrapolation distance (pooled, with 95% Clopper–Pearson bars)

In [ ]:
pred_all["d_bin"] = np.where(pred_all.d >= 4, "4+", pred_all.d.astype(str))
by_d = summarize(pred_all, by="d_bin")
order = [b for b in ["1", "2", "3", "4+"] if b in set(pred_all.d_bin)]
print(pred_all.d_bin.value_counts().reindex(order).rename("test nuclei"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
offs = np.linspace(-0.2, 0.2, len(METHODS))
for j, m in enumerate(METHODS):
    q = by_d[by_d.method == m].set_index("d_bin").reindex(order)
    x = np.arange(len(order)) + offs[j]
    axes[0].errorbar(x, q.coverage, yerr=[q.coverage - q.CP95_lo, q.CP95_hi - q.coverage],
                     fmt="o", capsize=3, label=m)
    axes[1].plot(np.arange(len(order)), q.mean_width_MeV, marker="o", label=m)
axes[0].axhline(1 - ALPHA, color="k", ls="--", lw=1)
for ax in axes:
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order); ax.set_xlabel("extrapolation distance d")
axes[0].set_ylabel("coverage (95% CP interval)"); axes[0].set_ylim(0.3, 1.02)
axes[1].set_ylabel("mean 90% interval width (MeV)")
axes[0].legend(fontsize=8, loc="lower left"); plt.tight_layout(); plt.show()
by_d.round(3)

In [ ]:
# Peeling curves: calibration constant per distance, every fold
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for _, r in fold_info.iterrows():
    cd = [r[f"peel_c_d{d}"] for d in range(1, K_PEEL + 1)]
    ax.plot(range(1, K_PEEL + 1), cd, marker="o", label=f"{r.fold} (γ={r.gamma:.2f})")
ax.set_xlabel("distance from core d"); ax.set_ylabel("90% CQR score quantile (MeV)")
ax.set_title("Frontier peeling"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 9. Paired comparisons (exact McNemar)

On the same test nuclei, does Peel-CQR cover significantly more (or less) often than each competitor? `b` = covered by Peel-CQR only, `c` = covered by the competitor only.

In [ ]:
rows = []
for fold, g in pred_all.groupby("fold"):
    for m in METHODS:
        if m == "Peel-CQR" or f"{m}_cov" not in g or g[f"{m}_cov"].isna().all():
            continue
        p, b, c = mcnemar_p(g["Peel-CQR_cov"].astype(bool).values, g[f"{m}_cov"].astype(bool).values)
        rows.append(dict(fold=fold, comparison=f"Peel-CQR vs {m}", n=len(g), b_peel_only=b, c_other_only=c,
                         p_value=p, width_ratio=(g["Peel-CQR_hi"] - g["Peel-CQR_lo"]).mean()
                         / (g[f"{m}_hi"] - g[f"{m}_lo"]).mean()))
mcnemar = pd.DataFrame(rows)
mcnemar.round(4)

## 10. Point accuracy

rms of the base model and of the refined median prediction on the pooled test nuclei, by distance.

In [ ]:
pred_all["err_base2"] = pred_all.y_MeV ** 2
pred_all["err_ref2"] = (pred_all.y_MeV - pred_all.median_MeV) ** 2
rms_tab = pred_all.groupby(["fold", "d_bin"])[["err_base2", "err_ref2"]].mean().pow(0.5) \
    .rename(columns={"err_base2": "base_rms_MeV", "err_ref2": "refined_rms_MeV"}).reset_index()
rms_tab.round(3)

## 11. Robustness: subsampling (80% of training nuclei, without replacement)

Run on the most recent fold (2012→2016→2020); GP is skipped here for speed.

In [ ]:
main_fold = (2012, 2016, 2020)
n_tr_main = int((np.array([k in MEAS[main_fold[0]] for k in data.key]) & data.measured2020.values).sum())
rows = []
for r in range(N_SUBSAMPLE):
    rs = np.random.default_rng(1000 + r)
    sub = np.sort(rs.choice(n_tr_main, int(0.8 * n_tr_main), replace=False))
    p, i, _ = run_fold(*main_fold, run_gp=False, sub_idx=sub, verbose=False)
    row = {"rep": r, "gamma": i["gamma"]}
    for m in ["CQR-random", "CQR-temporal", "CQR-temporal-scaled", "Peel-CQR"]:
        if f"{m}_cov" in p:
            row[f"cov_{m}"] = p[f"{m}_cov"].mean()
            row[f"width_{m}"] = (p[f"{m}_hi"] - p[f"{m}_lo"]).mean()
    rows.append(row)
robust = pd.DataFrame(rows)
robust_summary = robust.drop(columns="rep").agg(["mean", "std"]).T
robust_summary.round(3)

## 12. Sensitivity to the number of peeled layers K (most recent fold)

In [ ]:
rows = []
for K in [2, 3, 4, 5]:
    K_PEEL_SAVE = K_PEEL
    globals()["K_PEEL"] = K
    p, i, _ = run_fold(*main_fold, run_gp=False, verbose=False)
    globals()["K_PEEL"] = K_PEEL_SAVE
    rows.append(dict(K=K, gamma=i["gamma"], coverage=p["Peel-CQR_cov"].mean(),
                     width=(p["Peel-CQR_hi"] - p["Peel-CQR_lo"]).mean(),
                     cov_scaled=p.get("CQR-temporal-scaled_cov", pd.Series(dtype=float)).mean()))
sens_K = pd.DataFrame(rows)
sens_K.round(3)

## 13. Final table for unmeasured nuclei

Learner fitted on **all** nuclei measured in AME2020; Peel-CQR calibrated on its own frontier; distance measured from all measured nuclei. Columns give the median prediction and the 90% interval of the total binding energy (MeV). `beyond_calibrated_range` marks d > K, where the interval relies on the linear extension of the peeling curve.

In [ ]:
allm = data.measured2020.values
yv_all = (data.B.values - base_prediction(allm)) / 1000.0
q_final = QuantileGB().fit(X_ALL[allm], yv_all[allm])
peel_final = learn_peel(X_ALL[allm], yv_all[allm], ZN[allm])
fr = ~allm
d_fr = manhattan_to(ZN[allm], ZN[fr])
lo, med, hi = q_final.predict3(X_ALL[fr]); cpf = peel_c(peel_final, d_fr)
base = base_prediction(allm)[fr] / 1000.0
frontier = pd.DataFrame({"Z": data.Z.values[fr], "N": data.N.values[fr], "A": data.A.values[fr], "d": d_fr,
                         "B_median_MeV": base + med, "B_lo90_MeV": base + lo - cpf, "B_hi90_MeV": base + hi + cpf,
                         "B_AME2020_systematics_MeV": data.B.values[fr] / 1000.0,
                         "beyond_calibrated_range": d_fr > K_PEEL}).sort_values(["Z", "N"])
frontier["width_MeV"] = frontier.B_hi90_MeV - frontier.B_lo90_MeV
frontier["AME_inside"] = (frontier.B_AME2020_systematics_MeV >= frontier.B_lo90_MeV) & \
                         (frontier.B_AME2020_systematics_MeV <= frontier.B_hi90_MeV)
frontier.to_csv("predicted_masses_intervals_v2.csv", index=False)
print(f"final peel: core {peel_final['n_core']}, c_d {np.round(peel_final['c_d'], 3)}, slope {peel_final['slope']:.3f}")
print(f"{len(frontier)} unmeasured nuclei; {frontier.beyond_calibrated_range.sum()} beyond d={K_PEEL}")
print("agreement with AME2020 systematic estimates (not a coverage test):")
print(frontier.assign(dbin=np.where(frontier.d >= 6, "6+", frontier.d.astype(str)))
      .groupby("dbin").agg(n=("Z", "size"), width=("width_MeV", "mean"), inside=("AME_inside", "mean")).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(data.N[allm], data.Z[allm], s=3, c="#ced4da", marker="s", label="measured (AME2020)")
sc = ax.scatter(frontier.N, frontier.Z, s=6, c=frontier.width_MeV, cmap="viridis", marker="s")
plt.colorbar(sc, label="90% interval width (MeV)")
ax.set_xlabel("N"); ax.set_ylabel("Z"); ax.set_title("Peel-CQR interval width on unmeasured nuclei")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

## 14. Save everything to Excel

In [ ]:
try:
    import openpyxl  # noqa: F401
except ImportError:
    !pip install -q openpyxl

XLSX = "nuclear_mass_results_v2.xlsx"
config = pd.DataFrame({"parameter": ["alpha", "target coverage", "K_PEEL", "editions", "folds", "base model",
                                     "learner", "GP run", "subsample reps"],
                       "value": [ALPHA, 1 - ALPHA, K_PEEL, str(EDITIONS), str(FOLDS),
                                 "LDM (refit per fold)" if BASE_TABLE is None else "user table",
                                 "HistGradientBoosting quantiles", RUN_GP, N_SUBSAMPLE]})
sheets = {"config": config, "pooled_summary": pooled, "per_fold_summary": per_fold,
          "coverage_by_distance": by_d, "fold_info_peeling": fold_info, "mcnemar_tests": mcnemar,
          "point_rms": rms_tab, "robust_subsampling": robust, "robust_summary": robust_summary.reset_index(),
          "sensitivity_K": sens_K,
          "test_predictions": pred_all.drop(columns=["err_base2", "err_ref2"]),
          "frontier_predictions": frontier}
with pd.ExcelWriter(XLSX, engine="openpyxl") as xw:
    for name, df in sheets.items():
        df.round(4).to_excel(xw, sheet_name=name[:31], index=False)
    for ws in xw.book.worksheets:
        for col in ws.columns:
            w = max(len(str(c.value)) if c.value is not None else 0 for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(10, w + 2), 40)
        ws.freeze_panes = "A2"
print("saved", XLSX, "->", list(sheets))

## 15. Reading the results

* **Pooled table:** a method is calibrated if 0.90 lies inside its Clopper–Pearson interval (equivalently `p_vs_0.90` > 0.05). Under-coverage of CQR-random and GP is the evidence of covariate shift.
* **By distance:** the claim of the paper is that Peel-CQR keeps coverage near 0.90 at every d while CQR-random/CQR-temporal drift down as d grows.
* **McNemar:** significant `p_value` with `b > c` means Peel-CQR covers strictly more test nuclei; check `width_ratio` for the price paid in width.
* **Next:** load WS4 or FRDM2012 as `BASE_TABLE`, and plug your NDT network in place of `QuantileGB`.